## *RaschPy* simulation functionality

This notebook works through examples of how to generate simulated data sets with `RaschPy` for experimental use where knowledge of the underlying 'ground truth' of the generating parameters is useful, for example when comparing the efficacy of different estimation algorithms, such as in Elliott & Buttery (2022a) or exploring the effect of fitting different Rasch models to the same data set, such as in Elliott & Buttery (2022b). There are separate classes for each model: `SLM_Sim` for the simple logistic model (or dichotomous Rasch model) (Rasch, 1960), `PCM_Sim` for the partial credit model (Masters, 1982), `RSM_Sim` for the rating scale model (Andrich, 1978), `MFRM_Sim_Global` for the many-facet Rasch model (Linacre, 1994), and the family of extended MFRMs (Elliott 2025, Elliott & Buttery, 2022b): `MFRM_Sim_Items` for the vector-by-item extended MFRM , `MFRM_Sim_Thresholds` for the vector-by-threshold extended MFRM, `MFRM_Sim_Matrix` for the matrix extended MFRM, and `MFRM_Sim_Bivector` for the bivector extended MFRM. All data is generated to fit the chosen model.

**References**

&nbsp;&nbsp;&nbsp;&nbsp; Andrich, D. (1978). A rating formulation for ordered response categories. *Psychometrika*, *43*(4), 561–573.

&nbsp;&nbsp;&nbsp;&nbsp;   Elliott, M. (2025). Extended many-facet Rasch models: Accounting for rater effects in automated essay scoring systems [Apollo - University of Cambridge Repository]. https://doi.org/10.17863/CAM.127567

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M., & Buttery, P. J. (2022a) Non-iterative Conditional Pairwise Estimation for the Rating Scale Model, *Educational and Psychological Measurement*, *82*(5), 989-1019.

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M. and Buttery, P. J. (2022b) Extended Rater Representations in the Many-Facet Rasch Model, *Journal of Applied Measurement*, *22*(1), 133-160.

&nbsp;&nbsp;&nbsp;&nbsp; Linacre, J. M. (1994). *Many-Facet Rasch Measurement*. MESA Press.

&nbsp;&nbsp;&nbsp;&nbsp; Masters, G. N. (1982). A Rasch model for partial credit scoring. *Psychometrika*, *47*(2), 149–174.

&nbsp;&nbsp;&nbsp;&nbsp; Rasch, G. (1960). *Probabilistic models for some intelligence and attainment tests*. Danmarks Pædagogiske
Institut.

Import the packages and set the working directory (here called `my_working_directory`) - you will save your output files here.

In [ ]:
import raschpy as rp
import numpy as np
import pandas as pd
import os

os.chdir('my_working_directory')

### `MFRM_Sim_Items`

Create an object `mfrm_sim_1` of the class `MFRM_Sim_Items` with randomised item locations, shared threshold set and person locations. `MFRM_Sim_Items` will do this automatically when you pass `item_range`, `facet_range`, `category_base`, `max_disorder`, `person_sd` and `offset` arguments to the simulation: item locations and rater facet effects will be sampled from a uniform distribution; person locations will be sampled from a normal distribution. We pass `item_range=4` to have items covering a range of 4 logits, `facet_range=3` to have raters covering a range of 3 logits, and `person_sd=2` and `offset=1` to have a sample of persons with a mean location 1 logit higher than the items, with a standard deviation of 2 logits. We also pass the additional arguments `category_base=1.5` and `max_disorder=1`; this sets the base category width to 1.5 logits, with a degree of random uniform variation around controlled by `max_disorder`. With `max_disorder=1`, the minimum category width is 1 logit (and the maximum, symmetrically, will be 2 logits); a smaller value permits more variation in category widths, and a negative value for `max_disorder` allows the presence of disordered thresholds (hence the name of the argument). From this, a set of central item locations are generated from `item_range`, and a set of centred Rasch-Andrich thresholds, summing to zero, are generated from `category_base` and `max_disorder`. One other additional argument that must be passed to `MFRM_Sim_Items` is `max_score`, which is the maximum possible score for each item. There are 500 persons, 8 items and 10 raters, with no missing data for this simulation.

In [ ]:
mfrm_sim_1 = rp.MFRM_Sim_Items(no_of_items=8,
                               no_of_persons=500,
                               no_of_facet_elements=10,
                               max_score=5,
                               item_range=4,
                               facet_range=3,
                               category_base=1.5,
                               max_disorder=1,
                               person_sd=2,
                               offset=0.5)

Save the generated response dataframe, which is stored as an attribute `mfrm_sim_1.responses`, to file, and view the first 5 lines.

In [ ]:
mfrm_sim_1.responses.to_csv('mfrm_sim_1_responses.csv')
mfrm_sim_1.responses.head()

Save the generating item, threshold, rater and person parameters to file, and view the first 5 lines of the item locations, rater facet effects and person locations, plus the Rasch-Andrich thresholds.

In [ ]:
mfrm_sim_1.items.to_csv('mfrm_sim_1_items.csv', header=None)
mfrm_sim_1.items.head()

In [ ]:
mfrm_sim_1.thresholds.to_csv('mfrm_sim_1_thresholds.csv', header=None)
mfrm_sim_1.thresholds

In [ ]:
mfrm_sim_1.facet_effects.to_csv('mfrm_sim_1_facet_effects.csv')
mfrm_sim_1.facet_effects.head()

In [ ]:
mfrm_sim_1.persons.to_csv('mfrm_sim_1_persons.csv', header=None)
mfrm_sim_1.persons.head()

View `max_score`.

In [ ]:
mfrm_sim_1.max_score

Create an object `mfrm_1` of the class `MFRM` from the response dataframe for analysis. The new object `mfrm_1` automatically inherits all the parameters from `mfrm_sim_1`, storing them under a namespace `.generating`.

In [ ]:
mfrm_1 = rp.MFRM(mfrm_sim_1)

You may wish to create a simulation based on specified, known item locations and/or person locations. This may be done by passing lists to the `manual_items`, `manual_raters`, `manual_thresholds` and/or `manual_persons` arguments (in which case, there is no need to pass the relevant `item_range`, `category_base`, `max_disorder`, `person_sd` or `offset` arguments). You may also customise the names of the items and/or persons by passing lists of the correct length to the manual_person_names and/or manual_item_names arguments.

The manual_items and manual_persons arguments may also be used to generate random item locations and/or person locations according to distributions other than the default uniform (for items) and normal (for persons). This is what is done in the example `mfrm_sim_2` below: A set of specified, fixed item locations (6 items of location between -2.5 logit and +2.5 logits and a maximum score of 5) and a set of Rasch-Andrich thresholds (summing to zero) are passed together with 5 raters with specified facet-effect profiles and a random uniform distribution of person locations (between -2 and +2 logits). For this simulation, we also set a proportion of 10% missing data (missing completely at random) by passing the argument `missing=0.1`.

In [ ]:
mfrm_sim_2 = rp.MFRM_Sim_Items(no_of_items=6,
                               no_of_persons=500,
                               no_of_facet_elements=5,
                               max_score=5,
                               missing=0.1,
                               manual_persons = np.random.uniform(-2, 2, 500),
                               manual_items=[-2.5, -1.5, -0.5, 0.5, 1.5, 2.5],
                               manual_thresholds=[-2, -1, 0, 1, 2],
                               manual_raters = {'Rater_1': {'Item_1': 0, 'Item_2': 0, 'Item_3': 0, 'Item_4': 0, 'Item_5': 0, 'Item_6': 0},
                                                'Rater_2': {'Item_1': 0, 'Item_2': 0, 'Item_3': 0, 'Item_4': 0, 'Item_5': 0, 'Item_6': 0},
                                                'Rater_3': {'Item_1': -1, 'Item_2': -2, 'Item_3': 0, 'Item_4': 2, 'Item_5': -2, 'Item_6': -1},
                                                'Rater_4': {'Item_1': 2, 'Item_2': 1, 'Item_3': -1, 'Item_4': 2, 'Item_5': 1, 'Item_6': -1},
                                                'Rater_5': {'Item_1': 1, 'Item_2': 1, 'Item_3': 2, 'Item_4': 1, 'Item_5': 1, 'Item_6': 1}})

Save the generated response dataframe, which is stored as an attribute `mfrm_sim_2.responses`, to file, and view the first 5 lines.

In [ ]:
mfrm_sim_2.responses

Save the generating item, threshold and person parameters to file, and view the item locations and Rasch-Andrich thresholds.

In [ ]:
mfrm_sim_2.items.to_csv('mfrm_sim_2_items.csv', header=None)
mfrm_sim_2.items

In [ ]:
mfrm_sim_2.thresholds.to_csv('mfrm_sim_2_thresholds.csv', header=None)
mfrm_sim_2.thresholds

In [ ]:
mfrm_sim_2.facet_effects.to_csv('mfrm_sim_2_facet_effects.csv')
mfrm_sim_2.facet_effects.head()

In [ ]:
mfrm_sim_2.persons.to_csv('mfrm_sim_2_persons.csv', header=None)
mfrm_sim_2.persons.head()

View `max_score`.

In [ ]:
mfrm_sim_2.max_score

Create an object, `mfrm_2`, of the class `MFRM` from the response dataframe for analysis.

In [ ]:
mfrm_2 = rp.MFRM(mfrm_sim_2)

The two `MFRM` objects `mfrm_1` and `mfrm_2` are now available for analysis and, where appropriate, comparison of the recovered estmates with the generating estimates. See the example `Items MFRM` notebook for details on how to run an `MFRM` analysis.